# VeriteFiscale-2027 — Usine OCR Lourde (Google Colab Pro A100/L4)

**Rôle strict dans l'architecture :** Ce notebook sert UNIQUEMENT à convertir les annexes budgétaires massives (PLF/PLFSS 2027, rapports de 500 à 1 500+ pages, PDF scannés ou tableaux multi-colonnes) en Markdown/JSON paginé vers `data/programs/`.

**Ne jamais exécuter OpenFisca ici** : `OpenFisca-France` tourne localement en < 2 secondes sur le CPU ARM64 du RTX Spark sans GPU.

In [ ]:
#@title 1. Vérification GPU Colab Pro & Montage Google Drive
!nvidia-smi
from google.colab import drive
drive.mount('/content/drive')

import os
INPUT_DIR = '/content/drive/MyDrive/verite-fiscale-2027/raw_pdfs'
OUTPUT_DIR = '/content/drive/MyDrive/verite-fiscale-2027/extracted_programs'
os.makedirs(INPUT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'PDFs disponibles : {os.listdir(INPUT_DIR)}')

In [ ]:
#@title 2. Installation du moteur OCR choisi (Marker v2 ou MinerU 4.0)
ENGINE = 'marker'  #@param ['marker', 'mineru']

if ENGINE == 'marker':
    !pip install -q marker-pdf
else:
    !pip install -q magic-pdf[full]

In [ ]:
#@title 3. Conversion paginée avec traçabilité stricte (page_number + bbox)
import json
from pathlib import Path

def export_paginated_chunks(pdf_path: Path, out_dir: Path, engine: str = 'marker'):
    out_file = out_dir / f'{pdf_path.stem}.pages.jsonl'
    if engine == 'marker':
        from marker.converters.pdf import PdfConverter
        from marker.models import create_model_dict
        from marker.output import text_from_rendered
        converter = PdfConverter(artifact_dict=create_model_dict(), config={'paginate_output': True})
        rendered = converter(str(pdf_path))
        text, _, images = text_from_rendered(rendered)
        # Marker insère des séparateurs de page {N}------------------------------------------------
        pages = text.split('\n\n{')
        with open(out_file, 'w', encoding='utf-8') as f:
            for idx, chunk in enumerate(pages, start=1):
                record = {
                    'source_file': pdf_path.name,
                    'page': idx,
                    'text': chunk.strip(),
                    'ocr_engine': 'marker-v2'
                }
                f.write(json.dumps(record, ensure_ascii=False) + '\n')
    else:
        # MinerU CLI batch
        import subprocess
        subprocess.run(['magic-pdf', '-p', str(pdf_path), '-o', str(out_dir), '-m', 'auto'], check=True)
    print(f'Export terminé : {out_file}')

for pdf in Path(INPUT_DIR).glob('*.pdf'):
    export_paginated_chunks(pdf, Path(OUTPUT_DIR), ENGINE)